# Домашнее задание: Semi-Supervised Learning с Multi-Branch MLP

In [1]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

from model import MultiBranchMLP
from data_module import SemiSupervisedDataModule
from lightning_module import SemiSupervisedLightningModule

from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    import random
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


## 1. Загрузка данных


In [2]:
data_dir = './data'

dm = SemiSupervisedDataModule(
    data_dir=data_dir,
    batch_size=512,
    num_workers=8
)

dm.setup()

print(f'Input dimension: {dm.input_dim}')
print(f'Number of classes: {dm.n_classes}')
print(f'Labeled train samples: {len(dm.train_labeled_dataset)}')
print(f'Test samples: {len(dm.test_dataset)}')


Loading labeled dataset...
Loading unlabeled dataset...
Loading test dataset...
Input dimension: 3072
Number of classes: 10
Labeled train samples: 1280
Validation samples: 320
Unlabeled train samples: 14400
Test samples: 4000
Input dimension: 3072
Number of classes: 10
Labeled train samples: 1280
Test samples: 4000


## 2. Анализ дисбаланса классов и вычисление весов


In [3]:
train_labels = dm.train_labeled_dataset.y.numpy()

unique_labels = np.unique(train_labels)
class_weights = compute_class_weight(
    'balanced',
    classes=unique_labels,
    y=train_labels
)

print(f'Class weights: {dict(zip(unique_labels, class_weights))}')

class_weights_tensor = torch.FloatTensor(class_weights)


Class weights: {np.int64(0): np.float64(1.0756302521008403), np.int64(1): np.float64(0.9624060150375939), np.int64(2): np.float64(1.024), np.int64(3): np.float64(1.103448275862069), np.int64(4): np.float64(0.9552238805970149), np.int64(5): np.float64(0.9343065693430657), np.int64(6): np.float64(0.9481481481481482), np.int64(7): np.float64(1.0578512396694215), np.int64(8): np.float64(0.927536231884058), np.int64(9): np.float64(1.0491803278688525)}


## 3. Создание модели


In [4]:
model = MultiBranchMLP(
    input_dim=dm.input_dim,
    hidden_dim=128,
    output_dim=dm.n_classes,
    num_blocks=4,
    dropout=0.1,
    combine_mode='concat'
)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')


Model parameters: 1,231,626


## 4. Создание Lightning модуля


In [5]:
loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)

lightning_model = SemiSupervisedLightningModule(
    model=model,
    loss_fn=loss_fn,
    optimizer_type='adamw',
    learning_rate=1e-3,
    task_type='multiclass'
)


## 5. Обучение модели


In [6]:
checkpoint_callback = ModelCheckpoint(
    dirpath='checkpoints',
    filename='best_model-{epoch:02d}-{val_accuracy:.4f}',
    monitor='val_accuracy',
    mode='max',
    save_top_k=1,
    save_last=True
)

trainer = Trainer(
    max_epochs=10,
    callbacks=[checkpoint_callback],
    enable_checkpointing=True,
    logger=True,
    enable_progress_bar=True,
    enable_model_summary=True,
    accelerator='gpu',
    devices=1,
    precision='16-mixed', 
    log_every_n_steps=10
)

trainer.fit(lightning_model, dm)


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


Loading labeled dataset...
Loading unlabeled dataset...
Loading test dataset...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type             | Params | Mode  | FLOPs
-------------------------------------------------------------
0 | model   | MultiBranchMLP   | 1.2 M  | train | 0    
1 | loss_fn | CrossEntropyLoss | 0      | train | 0    
2 | metrics | ModuleDict       | 0      | train | 0    
-------------------------------------------------------------
1.2 M     Trainable params
0         Non-trainable params
1.2 M     Total params
4.927     Total estimated model params size (MB)
100       Modules in train mode
0         Modules in eval mode
0         Total Flops


Input dimension: 3072
Number of classes: 10
Labeled train samples: 1280
Validation samples: 320
Unlabeled train samples: 14400
Test samples: 4000
Epoch 9: 100%|██████████| 15/15 [00:01<00:00, 12.88it/s, v_num=3, train_pseudo_labels_used_step=0.000, train_supervised_loss_step=0.0216, train_total_loss_step=0.0216, val_loss=2.890, train_pseudo_labels_used_epoch=0.000, train_supervised_loss_epoch=0.0194, train_total_loss_epoch=0.0194]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 15/15 [00:02<00:00,  7.28it/s, v_num=3, train_pseudo_labels_used_step=0.000, train_supervised_loss_step=0.0216, train_total_loss_step=0.0216, val_loss=2.890, train_pseudo_labels_used_epoch=0.000, train_supervised_loss_epoch=0.0194, train_total_loss_epoch=0.0194]


## 6. Оценка на тестовой выборке


In [7]:
best_model_path = checkpoint_callback.best_model_path
print(f'Loading best model from: {best_model_path}')

if best_model_path:
    best_model = SemiSupervisedLightningModule.load_from_checkpoint(
        best_model_path,
        model=model,
        loss_fn=loss_fn,
        optimizer_type='adamw',
        learning_rate=1e-3,
        task_type='multiclass'
    )
else:
    best_model = lightning_model

test_results = trainer.test(best_model, dm)

print('\n=== Финальные результаты на тестовой выборке ===')
for key, value in test_results[0].items():
    print(f'{key}: {value:.4f}')


Loading best model from: C:\_MyGit\Deep_learning_autumn_2025\Deep_learning_MISIS\homework_3_semi_supervised_learning\checkpoints\best_model-epoch=09-val_accuracy=0.3812-v1.ckpt
Loading labeled dataset...
Loading unlabeled dataset...
Loading test dataset...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Input dimension: 3072
Number of classes: 10
Labeled train samples: 1280
Validation samples: 320
Unlabeled train samples: 14400
Test samples: 4000
Testing DataLoader 0: 100%|██████████| 4/4 [00:00<00:00, 13.77it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      test_accuracy         0.3569999933242798
      test_f1_macro         0.35692811012268066
        test_loss            3.03357195854187
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

=== Финальные результаты на тестовой выборке ===
test_loss: 3.0336
test_accuracy: 0.3570
test_f1_macro: 0.3569
